<a href="https://colab.research.google.com/github/iria-param/visitor_analytics/blob/codex%2Fapproach-2-id-stability/notebooks/colab_tracker_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Museum Gallery AI - Colab Tracker Comparison

Use this notebook to compare baseline ByteTrack, museum ByteTrack, and museum BoT-SORT on real experience-center CCTV clips.

Privacy rules:
- Do not commit videos, overlays, or events logs to Git.
- Keep clips in a private Drive folder.
- Run with `--no-overlay` for the batch comparison.
- Delete `/content/videos` and `/content/runs` after downloading comparison CSV/JSON.
- No ReID, face recognition, demographic inference, emotion inference, or person crops.

## 1. Enable GPU

In Colab: `Runtime -> Change runtime type -> T4 GPU` if available. Then run the next cell.

In [ ]:
!nvidia-smi
!python - <<'PY'
import torch
print('torch cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('GPU is not available. Change Colab runtime to GPU before continuing.')

Fri May 22 12:30:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Drive And Define Inputs

Upload the three CCTV clips to a private Google Drive folder first. Use stable file names with no spaces if possible.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this folder to your private Drive folder containing the CCTV clips.
DRIVE_VIDEO_DIR = '/content/drive/MyDrive/Visitor analysis'

# Edit these three clip entries after uploading the actual files.
CLIPS = [
    {'clip_id': 'gallery_day1', 'path': f'{DRIVE_VIDEO_DIR}/20260507140240.mp4'},
    # {'clip_id': 'gallery_day2', 'path': f'{DRIVE_VIDEO_DIR}/20260507140651.mp4'},
    # {'clip_id': 'gallery_day3', 'path': f'{DRIVE_VIDEO_DIR}/20260507140752.mp4'},
]

for clip in CLIPS:
    print(clip['clip_id'], clip['path'])

Mounted at /content/drive
gallery_day1 /content/drive/MyDrive/Visitor analysis/20260507140240.mp4


## 3. Clone And Install Project

Set `REPO_URL` to your GitHub repository URL if this project is pushed. If not, upload a zip of the project and adjust this cell.

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/iria-param/visitor_analytics.git'
BRANCH = 'codex/approach-2-id-stability'
PROJECT_DIR = Path('/content/museum-gallery-ai')

if REPO_URL == 'PASTE_YOUR_REPO_URL_HERE':
    raise ValueError('Set REPO_URL to the GitHub repository URL before continuing.')

if not PROJECT_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    print('Project already exists:', PROJECT_DIR)
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

!cd {PROJECT_DIR} && git log --oneline -3

os.chdir(PROJECT_DIR)

!python -m pip install --upgrade pip
!python -m pip install -r requirements-dev.txt
!python -m pip install -e .
!python -m pytest -v

Cloning into '/content/museum-gallery-ai'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 154 (delta 59), reused 134 (delta 44), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 150.30 KiB | 6.83 MiB/s, done.
Resolving deltas: 100% (59/59), done.
764a91c (HEAD -> codex/approach-2-id-stability, origin/codex/approach-2-id-stability) docs: route notebooks/ in CLAUDE.md; log notebook consolidation + M1+M2-prep; commit research 0005
7a2aca7 Approach 2 M1+M2-prep: pin Ultralytics; expose detector.iou as config + CLI knob
eb3e052 Consolidate Colab notebook to notebooks/; remove stray approach-2-id-stability/ duplicate
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.1 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# M2 experiment: NMS iou=0.4 (current default) vs iou=0.5 on gallery_day1.
# Same tracker (museum BoT-SORT, no ReID), same clip, same frame budget. Only iou differs.
# Per research 0005, this targets the proximity-merge case (F1).

import json, subprocess, time
from pathlib import Path

clip = CLIPS[0]  # gallery_day1; needs section 2 to have run
m2_runs_dir = Path('/content/m2_runs')
m2_runs_dir.mkdir(parents=True, exist_ok=True)
runtime_config_dir = Path('/content/runtime_configs')
runtime_config_dir.mkdir(parents=True, exist_ok=True)

# Build M2 base config: museum BoT-SORT (no ReID), 1800 frames, overlay on, GPU.
base = json.loads(Path('/content/museum-gallery-ai/configs/eval/tracker_only.json').read_text())
base['detector']['tracker'] = 'configs/trackers/botsort_museum.yaml'
base['detector']['device'] = 'cuda'
base['detector']['image_size'] = 1280
base['processing']['max_frames'] = 1800
base['processing']['frame_stride'] = 1
base['processing']['write_overlay'] = True

m2_config = runtime_config_dir / 'tracker_only_M2_base.json'
m2_config.write_text(json.dumps(base, indent=2))
print('M2 base config written to', m2_config)

for iou_label, iou_value in [('iou_040_baseline', 0.4), ('iou_050_M2', 0.5)]:
    run_name = f"{clip['clip_id']}_{iou_label}"
    output_dir = m2_runs_dir / run_name
    done_file = output_dir / '_done.txt'
    if done_file.exists():
        print('Skipping completed run:', run_name)
        continue
    output_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', '-m', 'museum_gallery_ai', 'process',
        '--config', str(m2_config),
        '--source', str(clip['path']),
        '--output', str(output_dir),
        '--detector-iou', str(iou_value),
    ]
    print('Running:', ' '.join(cmd))
    started = time.perf_counter()
    subprocess.run(cmd, check=True)
    elapsed = time.perf_counter() - started
    done_file.write_text(f"completed in {elapsed:.2f}s\n")
    print(f'Completed: {run_name} in {elapsed:.1f}s')

print('\n--- Comparison ---')
!python scripts/compare_runs.py --runs {m2_runs_dir} --out {m2_runs_dir}/comparison_M2.csv --expected-runs 2

import pandas as pd
df = pd.read_csv(m2_runs_dir / 'comparison_M2.csv')
display(df)

M2 base config written to /content/runtime_configs/tracker_only_M2_base.json
Running: python -m museum_gallery_ai process --config /content/runtime_configs/tracker_only_M2_base.json --source /content/drive/MyDrive/Visitor analysis/20260507140240.mp4 --output /content/m2_runs/gallery_day1_iou_040_baseline --detector-iou 0.4
Completed: gallery_day1_iou_040_baseline in 152.5s
Running: python -m museum_gallery_ai process --config /content/runtime_configs/tracker_only_M2_base.json --source /content/drive/MyDrive/Visitor analysis/20260507140240.mp4 --output /content/m2_runs/gallery_day1_iou_050_M2 --detector-iou 0.5
Completed: gallery_day1_iou_050_M2 in 139.1s

--- Comparison ---
Wrote 2 rows to /content/m2_runs/comparison_M2.csv
Wrote 2 rows to /content/m2_runs/comparison_M2.json


,clip_id,tracker,unique_track_count,real_track_count,fallback_track_count,duration_median,duration_p25,duration_p75,duration_max,short_lived_count,likely_switch_count,total_gaps,max_gap_processed_frames,tracks_per_minute,run_seconds,track_buffer
0,gallery_day1_iou_040,baseline,48,40,8,2.86,0.56,8.68,47.64,11,16,81,83,33.352,148.475,90
1,gallery_day1_iou_050_M2,unknown,48,40,8,3.26,0.90,6.94,47.64,10,15,79,83,33.352,137.907,90


In [ ]:
from pathlib import Path

experimental_yaml = """\
# EXPERIMENTAL: BoT-SORT with appearance ReID enabled.
# Offline experiment only — not a deployment decision.
# See docs/AGENT_COMMUNICATION.md 2026-05-14 entry for boundary.

tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.30
track_buffer: 90
match_thresh: 0.8
fuse_score: True
gmc_method: sparseOptFlow

# --- ReID ENABLED for this experiment ---
proximity_thresh: 0.5
appearance_thresh: 0.8
with_reid: True
model: auto
"""

cfg_path = Path('/content/museum-gallery-ai/configs/trackers/botsort_museum_reid.yaml')
cfg_path.write_text(experimental_yaml)
print('Wrote', cfg_path)
print(cfg_path.read_text())

Wrote /content/museum-gallery-ai/configs/trackers/botsort_museum_reid.yaml
# EXPERIMENTAL: BoT-SORT with appearance ReID enabled.
# Offline experiment only — not a deployment decision.
# See docs/AGENT_COMMUNICATION.md 2026-05-14 entry for boundary.

tracker_type: botsort
track_high_thresh: 0.25
track_low_thresh: 0.1
new_track_thresh: 0.30
track_buffer: 90
match_thresh: 0.8
fuse_score: True
gmc_method: sparseOptFlow

# --- ReID ENABLED for this experiment ---
proximity_thresh: 0.5
appearance_thresh: 0.8
with_reid: True
model: auto



## 4. Generate Runtime Tracker Configs

These generated configs are runtime-only and should not be committed.

In [ ]:
import copy
import json
from pathlib import Path

base_config = json.loads(Path('configs/eval/tracker_only.json').read_text())
runtime_config_dir = Path('/content/runtime_configs')
runtime_config_dir.mkdir(parents=True, exist_ok=True)

TRACKERS = {
    'botsort_museum_reid': 'configs/trackers/botsort_museum_reid.yaml',
}

for tracker_name, tracker_path in TRACKERS.items():
    config = copy.deepcopy(base_config)
    config['detector']['tracker'] = tracker_path
    config['detector']['device'] = 'cuda'
    config['detector']['image_size'] = 1280
    config['processing']['frame_stride'] = 1
    config['processing']['write_overlay'] = True
    config['processing']['max_frames'] = 1800
    out_path = runtime_config_dir / f'tracker_only_{tracker_name}.json'
    out_path.write_text(json.dumps(config, indent=2))
    print(tracker_name, out_path)

botsort_museum_reid /content/runtime_configs/tracker_only_botsort_museum_reid.json


In [ ]:
!cat /content/runtime_configs/tracker_only_baseline.json | python -c "import sys,json;c=json.load(sys.stdin);print('write_overlay =',c['processing']['write_overlay'],'max_frames =',c['processing']['max_frames'])"

write_overlay = True max_frames = 1800


usage: museum_gallery_ai process [-h] --config CONFIG --source SOURCE --output
                                 OUTPUT [--max-frames MAX_FRAMES]
                                 [--frame-stride FRAME_STRIDE]
                                 [--image-size IMAGE_SIZE] [--no-overlay]

options:
  -h, --help            show this help message and exit
  --config CONFIG       Path to YAML config
  --source SOURCE       Path to recorded video file
  --output OUTPUT       Output directory for events, metrics, and overlay
  --max-frames MAX_FRAMES
                        Override processing.max_frames for quick evaluations
  --frame-stride FRAME_STRIDE
                        Override processing.frame_stride
  --image-size IMAGE_SIZE
                        Override detector.image_size
  --no-overlay          Skip overlay.mp4 rendering for faster metric runs


## 5. Run Batch Comparison

This runs 3 clips x 3 trackers. It writes `_done.txt` after each successful run so you can rerun the cell after a reconnect.

In [ ]:
import subprocess
import time
from pathlib import Path

runs_dir = Path('/content/runs')
runs_dir.mkdir(parents=True, exist_ok=True)

for clip in CLIPS:
    source = Path(clip['path'])
    if not source.exists():
        raise FileNotFoundError(source)
    for tracker_name in TRACKERS:
        run_name = f"{clip['clip_id']}_{tracker_name}"
        output_dir = runs_dir / run_name
        done_file = output_dir / '_done.txt'
        if done_file.exists():
            print('Skipping completed run:', run_name)
            continue
        output_dir.mkdir(parents=True, exist_ok=True)
        config_path = runtime_config_dir / f'tracker_only_{tracker_name}.json'
        command = [
            'python', '-m', 'museum_gallery_ai', 'process',
            '--config', str(config_path),
            '--source', str(source),
            '--output', str(output_dir)
        ]
        print('Running:', ' '.join(command))
        started = time.perf_counter()
        subprocess.run(command, check=True)
        done_file.write_text(f"completed in {time.perf_counter() - started:.2f}s\n")
        print('Completed:', run_name)

print('Batch complete.')

Running: python -m museum_gallery_ai process --config /content/runtime_configs/tracker_only_botsort_museum_reid.json --source /content/drive/MyDrive/Visitor analysis/20260507140240.mp4 --output /content/runs/gallery_day1_botsort_museum_reid
Completed: gallery_day1_botsort_museum_reid
Batch complete.


In [ ]:
!ls -la /content/runs/*/overlay.mp4

-rw-r--r-- 1 root root 99424337 May 14 13:23 /content/runs/gallery_day1_baseline/overlay.mp4
-rw-r--r-- 1 root root 99028172 May 14 13:27 /content/runs/gallery_day1_botsort_museum/overlay.mp4
-rw-r--r-- 1 root root 99111075 May 14 13:25 /content/runs/gallery_day1_bytetrack_museum/overlay.mp4


In [ ]:
!ls -la /content/runs/gallery_day1_botsort_museum_reid/

total 96800
drwxr-xr-x 2 root root     4096 May 15 16:02 .
drwxr-xr-x 3 root root     4096 May 15 15:59 ..
-rw-r--r-- 1 root root       21 May 15 16:02 _done.txt
-rw-r--r-- 1 root root        0 May 15 15:59 events.jsonl
-rw-r--r-- 1 root root     1350 May 15 16:01 metrics_summary.json
-rw-r--r-- 1 root root 99106425 May 15 16:01 overlay.mp4


In [ ]:
import shutil
from pathlib import Path

src = Path('/content/runs/gallery_day1_botsort_museum_reid/overlay.mp4')
dst = Path('/content/drive/MyDrive/Visitor analysis/overlays_visual_check/gallery_day1_botsort_museum_reid_overlay.mp4')
shutil.copy2(src, dst)
print('Copied to', dst)

Copied to /content/drive/MyDrive/Visitor analysis/overlays_visual_check/gallery_day1_botsort_museum_reid_overlay.mp4


In [ ]:
import shutil
from pathlib import Path

runs_dir = Path('/content/runs')
overlay_drive_dir = Path('/content/drive/MyDrive/Visitor analysis/overlays_visual_check')
overlay_drive_dir.mkdir(parents=True, exist_ok=True)

copied = 0
for overlay_path in runs_dir.glob('*/overlay.mp4'):
    destination = overlay_drive_dir / f"{overlay_path.parent.name}_overlay.mp4"
    shutil.copy2(overlay_path, destination)
    print('Copied', destination)
    copied += 1

print(f'Copied {copied} overlay files to {overlay_drive_dir}')

Copied /content/drive/MyDrive/Visitor analysis/overlays_visual_check/gallery_day1_bytetrack_museum_overlay.mp4
Copied /content/drive/MyDrive/Visitor analysis/overlays_visual_check/gallery_day1_botsort_museum_overlay.mp4
Copied /content/drive/MyDrive/Visitor analysis/overlays_visual_check/gallery_day1_baseline_overlay.mp4
Copied 3 overlay files to /content/drive/MyDrive/Visitor analysis/overlays_visual_check


## 6. Build Comparison CSV And JSON

In [ ]:
comparison_csv = runs_dir / 'comparison_real_cctv.csv'
!python scripts/compare_runs.py --runs {runs_dir} --out {comparison_csv} --expected-runs 9

import pandas as pd
df = pd.read_csv(comparison_csv)
display(df)
display(df.groupby('tracker')[['unique_track_count', 'duration_median', 'short_lived_count', 'likely_switch_count', 'max_gap_processed_frames']].median())

## 7. Download Only Comparison Outputs

In [ ]:
from google.colab import files

files.download(str(runs_dir / 'comparison_real_cctv.csv'))
files.download(str(runs_dir / 'comparison_real_cctv.json'))

## 8. Cleanup Runtime Files

Run this after downloading the comparison outputs.

In [ ]:
!rm -rf /content/runs /content/runtime_configs
print('Deleted runtime runs/configs from Colab VM. Disconnect and delete runtime next.')